In [ ]:
# Cell 1
!pip install numpy==1.26.4 -q
!pip install torch-geometric -q

In [ ]:
# Cell 2
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, GCNConv, global_mean_pool
import numpy as np
import pandas as pd
import os
import glob
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

DRIVE = '/content/drive/MyDrive/antibody_benchmark'
RESULTS_DIR = f'{DRIVE}/paegat_v1_results'
FIG_DIR = f'{DRIVE}/figures'
os.makedirs(FIG_DIR, exist_ok=True)

BOLTZ_DIR = f'{DRIVE}/paegat_dataset'
OF3_DIR = f'{DRIVE}/of3_paegat_dataset'
PROT_DIR = f'{DRIVE}/protenix_paegat_dataset'

def load_graphs(dataset_dir):
    files = glob.glob(f'{dataset_dir}/*.pt')
    graphs = []
    for f in files:
        try:
            graphs.append(torch.load(f, weights_only=False))
        except:
            pass
    print(f"Loaded {len(graphs)} from {dataset_dir.split('/')[-1]}")
    return graphs

def compute_ece(confidence, accuracy, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0
    for i in range(n_bins):
        mask = (confidence >= bins[i]) & (confidence < bins[i+1])
        if mask.sum() > 0:
            ece += mask.mean() * abs(confidence[mask].mean() - accuracy[mask].mean())
    return ece

print("Loading datasets...")
boltz_graphs = load_graphs(BOLTZ_DIR)
of3_graphs = load_graphs(OF3_DIR)
prot_graphs = load_graphs(PROT_DIR)

for g in boltz_graphs + of3_graphs + prot_graphs:
    if hasattr(g, 'seed_dir'):
        del g.seed_dir
    if hasattr(g, 'seed'):
        del g.seed

print(f"Total: Boltz-2={len(boltz_graphs)}, OF3={len(of3_graphs)}, Protenix={len(prot_graphs)}")

# Same train/test split as V1 Round 1
train_graphs = of3_graphs + prot_graphs
test_graphs = boltz_graphs
train_loader = DataLoader(train_graphs, batch_size=16, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=16, shuffle=False)

# ============================================================
# BASELINE 3: GCN (same graph, no attention, no edge features)
# ============================================================
class GCNBaseline(nn.Module):
    def __init__(self, in_channels=23, hidden_channels=64):
        super().__init__()
        self.input_proj = nn.Linear(in_channels, hidden_channels)
        self.conv1 = GCNConv(hidden_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.output = nn.Sequential(
            nn.Linear(hidden_channels, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x, edge_index, edge_attr, batch):
        x = F.relu(self.input_proj(x))
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.output(x).squeeze(-1)

# ============================================================
# BASELINE 4: GAT without edge features
# ============================================================
class GATNoEdge(nn.Module):
    def __init__(self, in_channels=23, hidden_channels=64, heads=4):
        super().__init__()
        self.input_proj = nn.Linear(in_channels, hidden_channels)
        self.conv1 = GATv2Conv(hidden_channels, hidden_channels // heads, heads=heads)
        self.conv2 = GATv2Conv(hidden_channels, hidden_channels // heads, heads=heads)
        self.output = nn.Sequential(
            nn.Linear(hidden_channels, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        self.dropout = nn.Dropout(0.2)

    def forward(self, x, edge_index, edge_attr, batch):
        x = F.relu(self.input_proj(x))
        x = F.relu(self.conv1(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.output(x).squeeze(-1)

def train_baseline(model, loader, epochs=30):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.MSELoss()
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            pred = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
            loss = criterion(pred, batch.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(loader):.4f}")
    return model

def evaluate_model(model, loader):
    model.eval()
    preds, labels, iptms = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
            preds.extend(pred.cpu().numpy())
            labels.extend(batch.y.cpu().numpy())
            iptms.extend(batch.iptm.cpu().numpy())
    return np.array(preds), np.array(labels), np.array(iptms)

# Train GCN
print("\n--- Baseline 3: GCN ---")
gcn = GCNBaseline().to(device)
gcn = train_baseline(gcn, train_loader, epochs=30)
preds_gcn, labels, iptms = evaluate_model(gcn, test_loader)
acceptable = (labels >= 0.23).astype(float)
ece_gcn = compute_ece(preds_gcn, acceptable)
r_gcn, _ = stats.spearmanr(preds_gcn, labels)
fpr_gcn = ((preds_gcn >= 0.5) & (labels < 0.23)).sum() / max((preds_gcn >= 0.5).sum(), 1)
print(f"GCN: ECE={ece_gcn:.4f}, r={r_gcn:.3f}, FPR={fpr_gcn:.3f}")

# Train GAT no edge
print("\n--- Baseline 4: GAT (no edge features) ---")
gat_no_edge = GATNoEdge().to(device)
gat_no_edge = train_baseline(gat_no_edge, train_loader, epochs=30)
preds_gat, labels, iptms = evaluate_model(gat_no_edge, test_loader)
ece_gat = compute_ece(preds_gat, acceptable)
r_gat, _ = stats.spearmanr(preds_gat, labels)
fpr_gat = ((preds_gat >= 0.5) & (labels < 0.23)).sum() / max((preds_gat >= 0.5).sum(), 1)
print(f"GAT (no edge): ECE={ece_gat:.4f}, r={r_gat:.3f}, FPR={fpr_gat:.3f}")

# ============================================================
# ATTENTION WEIGHT ANALYSIS
# ============================================================
print("\n--- Attention Weight Analysis ---")

class PAEGATWithAttention(nn.Module):
    def __init__(self, in_channels=23, hidden_channels=64, heads=4, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(in_channels, hidden_channels)
        self.conv1 = GATv2Conv(hidden_channels, hidden_channels // heads, heads=heads, edge_dim=1, dropout=dropout, return_attention_weights=True)  # Note: won't work directly
        self.conv2 = GATv2Conv(hidden_channels, hidden_channels // heads, heads=heads, edge_dim=1, dropout=dropout)
        self.output = nn.Sequential(
            nn.Linear(hidden_channels, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, edge_attr, batch, return_attention=False):
        x = F.relu(self.input_proj(x))
        if return_attention:
            x, (att_edge_index, att_weights) = self.conv1(x, edge_index, edge_attr, return_attention_weights=True)
            x = F.relu(x)
            x = self.dropout(x)
            x = F.relu(self.conv2(x, edge_index, edge_attr))
            x = global_mean_pool(x, batch)
            return self.output(x).squeeze(-1), att_edge_index, att_weights
        else:
            x = F.relu(self.conv1(x, edge_index, edge_attr))
            x = self.dropout(x)
            x = F.relu(self.conv2(x, edge_index, edge_attr))
            x = global_mean_pool(x, batch)
            return self.output(x).squeeze(-1)

# Load Round 1 weights
paegat = PAEGATWithAttention().to(device)
state_dict = torch.load(f'{RESULTS_DIR}/paegat_v1_round1.pt', map_location=device)
paegat.load_state_dict(state_dict)
paegat.eval()

# Analyse attention on test set
attention_results = []

for g in test_graphs[:100]:  # Sample 100 complexes
    try:
        data = g.to(device)
        heavy_len = g.heavy_len
        light_len = g.light_len
        antigen_len = g.antigen_len
        antigen_offset = heavy_len + light_len

        with torch.no_grad():
            pred, att_idx, att_w = paegat(
                data.x.unsqueeze(0) if data.x.dim() == 1 else data.x,
                data.edge_index,
                data.edge_attr,
                torch.zeros(data.x.shape[0], dtype=torch.long).to(device),
                return_attention=True
            )

        # Classify edges by interface type
        src = att_idx[0].cpu().numpy()
        dst = att_idx[1].cpu().numpy()
        weights = att_w.mean(dim=1).cpu().numpy()  # Average across heads

        ha_mask = ((src < heavy_len) & (dst >= antigen_offset)) | \
                  ((dst < heavy_len) & (src >= antigen_offset))
        la_mask = ((src >= heavy_len) & (src < antigen_offset) & (dst >= antigen_offset)) | \
                  ((dst >= heavy_len) & (dst < antigen_offset) & (src >= antigen_offset))
        hl_mask = ((src < heavy_len) & (dst >= heavy_len) & (dst < antigen_offset)) | \
                  ((dst < heavy_len) & (src >= heavy_len) & (src < antigen_offset))

        attention_results.append({
            'pdb': g.pdb,
            'mean_att_ha': weights[ha_mask].mean() if ha_mask.sum() > 0 else np.nan,
            'mean_att_la': weights[la_mask].mean() if la_mask.sum() > 0 else np.nan,
            'mean_att_hl': weights[hl_mask].mean() if hl_mask.sum() > 0 else np.nan,
            'paegat_pred': pred.item(),
            'dockq_ha': g.y.item(),
            'iptm': g.iptm.item()
        })
    except Exception as e:
        pass

df_att = pd.DataFrame(attention_results)
df_att.to_csv(f'{RESULTS_DIR}/paegat_attention_analysis.tsv', sep='\t', index=False)
print(f"Attention analysis: {len(df_att)} complexes")
print(f"Mean attention H/Ag: {df_att['mean_att_ha'].mean():.4f}")
print(f"Mean attention L/Ag: {df_att['mean_att_la'].mean():.4f}")
print(f"Mean attention H/L:  {df_att['mean_att_hl'].mean():.4f}")

# ============================================================
# FIGURES
# ============================================================
print("\n--- Generating Figures ---")

# Load V1 results
df_r1 = pd.read_csv(f'{RESULTS_DIR}/paegat_v1_round1_results.tsv', sep='\t')
df_r2 = pd.read_csv(f'{RESULTS_DIR}/paegat_v1_round2_results.tsv', sep='\t')
df_r3 = pd.read_csv(f'{RESULTS_DIR}/paegat_v1_round3_results.tsv', sep='\t')

# Figure 1: Reliability curves (3 panels)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
n_bins = 10
bins = np.linspace(0, 1, n_bins + 1)
bin_centers = (bins[:-1] + bins[1:]) / 2

for ax, df_r, title in zip(axes,
                            [df_r1, df_r2, df_r3],
                            ['Boltz-2 (Round 1)', 'OpenFold3 (Round 2)', 'Protenix (Round 3)']):
    labels = df_r['dockq_ha'].values
    preds = df_r['paegat_pred'].values
    iptms = df_r['iptm'].values
    acceptable = (labels >= 0.23).astype(float)

    paegat_acc, iptm_acc = [], []
    for i in range(n_bins):
        m_p = (preds >= bins[i]) & (preds < bins[i+1])
        m_i = (iptms >= bins[i]) & (iptms < bins[i+1])
        paegat_acc.append(acceptable[m_p].mean() if m_p.sum() > 0 else np.nan)
        iptm_acc.append(acceptable[m_i].mean() if m_i.sum() > 0 else np.nan)

    ax.plot([0,1],[0,1],'k--', label='Perfect', alpha=0.5)
    ax.plot(bin_centers, paegat_acc, 'b-o', label='PAE-GAT', linewidth=2)
    ax.plot(bin_centers, iptm_acc, 'r-s', label='ipTM', linewidth=2)
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Fraction Acceptable (DockQ≥0.23)')
    ax.set_title(title)
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_reliability_curves.pdf', dpi=300, bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/fig_reliability_curves.png', dpi=300, bbox_inches='tight')
print("Saved reliability curves")

# Figure 2: Ablation ladder bar chart
fig, ax = plt.subplots(figsize=(10, 6))
methods = ['ipTM\nbaseline', 'Random\nForest', 'GCN', 'GAT\n(no edge)', 'PAE-GAT\n(ours)']
ece_values = [0.3219, 0.2037, ece_gcn, ece_gat, 0.0478]
colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']

bars = ax.bar(methods, ece_values, color=colors, alpha=0.8, edgecolor='black')
ax.set_ylabel('Expected Calibration Error (ECE)', fontsize=12)
ax.set_title('Ablation Study — PAE-GAT Components (Boltz-2 test set)', fontsize=13)
for bar, val in zip(bars, ece_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0, 0.40)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_ablation_ladder.pdf', dpi=300, bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/fig_ablation_ladder.png', dpi=300, bbox_inches='tight')
print("Saved ablation ladder")

# Figure 3: Scatter plot PAE-GAT pred vs DockQ_HA (Round 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (conf, name, color) in zip(axes, [
    (df_r1['paegat_pred'].values, 'PAE-GAT', '#9467bd'),
    (df_r1['iptm'].values, 'ipTM', '#d62728')
]):
    labels = df_r1['dockq_ha'].values
    r, _ = stats.spearmanr(conf, labels)
    ax.scatter(conf, labels, alpha=0.3, s=10, color=color)
    ax.set_xlabel(f'{name} Score')
    ax.set_ylabel('DockQ_HA')
    ax.set_title(f'{name} vs DockQ_HA (Boltz-2)\nSpearman r = {r:.3f}')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.plot([0,1],[0.23,0.23],'k--',alpha=0.3, label='DockQ threshold')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_scatter_pred_vs_dockq.pdf', dpi=300, bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/fig_scatter_pred_vs_dockq.png', dpi=300, bbox_inches='tight')
print("Saved scatter plots")

# Figure 4: Attention by interface type
if len(df_att) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    interface_types = ['H/Ag\n(Heavy-Antigen)', 'L/Ag\n(Light-Antigen)', 'H/L\n(Heavy-Light)']
    att_means = [df_att['mean_att_ha'].mean(), df_att['mean_att_la'].mean(), df_att['mean_att_hl'].mean()]
    att_stds = [df_att['mean_att_ha'].std(), df_att['mean_att_la'].std(), df_att['mean_att_hl'].std()]
    colors = ['#d62728', '#1f77b4', '#2ca02c']
    bars = ax.bar(interface_types, att_means, yerr=att_stds, color=colors, alpha=0.8,
                  edgecolor='black', capsize=5)
    ax.set_ylabel('Mean Attention Weight', fontsize=12)
    ax.set_title('PAE-GAT Learned Attention by Interface Type\n(Boltz-2 test set, n=100 complexes)', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/fig_attention_by_interface.pdf', dpi=300, bbox_inches='tight')
    plt.savefig(f'{FIG_DIR}/fig_attention_by_interface.png', dpi=300, bbox_inches='tight')
    print("Saved attention figure")

print("\nAll figures saved to Drive.")
print(f"\n=== ABLATION SUMMARY ===")
print(f"ipTM baseline:        ECE = 0.3219")
print(f"Random Forest:        ECE = 0.2037")
print(f"GCN:                  ECE = {ece_gcn:.4f}")
print(f"GAT (no edge):        ECE = {ece_gat:.4f}")
print(f"PAE-GAT (ours):       ECE = 0.0478")

from google.colab import runtime
runtime.unassign()

Device: cuda
Loading datasets...
Loaded 1109 from paegat_dataset
Loaded 1129 from of3_paegat_dataset
Loaded 1162 from protenix_paegat_dataset
Total: Boltz-2=1109, OF3=1129, Protenix=1162

--- Baseline 3: GCN ---
Epoch 10/30 | Loss: 0.1040
Epoch 20/30 | Loss: 0.1030
Epoch 30/30 | Loss: 0.1019
GCN: ECE=0.1087, r=0.014, FPR=0.000

--- Baseline 4: GAT (no edge features) ---
Epoch 10/30 | Loss: 0.1039
Epoch 20/30 | Loss: 0.1014
Epoch 30/30 | Loss: 0.0972
GAT (no edge): ECE=0.1043, r=0.071, FPR=0.293

--- Attention Weight Analysis ---


TypeError: MessagePassing.__init__() got an unexpected keyword argument 'return_attention_weights'

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, global_mean_pool
import numpy as np
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from scipy import stats

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DRIVE = '/content/drive/MyDrive/antibody_benchmark'
RESULTS_DIR = f'{DRIVE}/paegat_v1_results'
FIG_DIR = f'{DRIVE}/figures'
os.makedirs(FIG_DIR, exist_ok=True)

BOLTZ_DIR = f'{DRIVE}/paegat_dataset'

def load_graphs(dataset_dir):
    files = glob.glob(f'{dataset_dir}/*.pt')
    graphs = []
    for f in files:
        try:
            graphs.append(torch.load(f, weights_only=False))
        except:
            pass
    return graphs

def compute_ece(confidence, accuracy, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0
    for i in range(n_bins):
        mask = (confidence >= bins[i]) & (confidence < bins[i+1])
        if mask.sum() > 0:
            ece += mask.mean() * abs(confidence[mask].mean() - accuracy[mask].mean())
    return ece

print("Loading Boltz-2 test graphs...")
boltz_graphs = load_graphs(BOLTZ_DIR)
for g in boltz_graphs:
    if hasattr(g, 'seed'):
        del g.seed
    if hasattr(g, 'seed_dir'):
        del g.seed_dir
print(f"Loaded {len(boltz_graphs)} Boltz-2 graphs")

# PAE-GAT with attention
class PAEGATWithAttention(nn.Module):
    def __init__(self, in_channels=23, hidden_channels=64, heads=4, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(in_channels, hidden_channels)
        self.conv1 = GATv2Conv(hidden_channels, hidden_channels // heads, heads=heads, edge_dim=1, dropout=dropout)
        self.conv2 = GATv2Conv(hidden_channels, hidden_channels // heads, heads=heads, edge_dim=1, dropout=dropout)
        self.output = nn.Sequential(
            nn.Linear(hidden_channels, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, edge_attr, batch, return_attention=False):
        x = F.relu(self.input_proj(x))
        if return_attention:
            out = self.conv1(x, edge_index, edge_attr, return_attention_weights=True)
            x, (att_edge_index, att_weights) = out
            x = F.relu(x)
            x = self.dropout(x)
            x = F.relu(self.conv2(x, edge_index, edge_attr))
            x = global_mean_pool(x, batch)
            return self.output(x).squeeze(-1), att_edge_index, att_weights
        else:
            x = F.relu(self.conv1(x, edge_index, edge_attr))
            x = self.dropout(x)
            x = F.relu(self.conv2(x, edge_index, edge_attr))
            x = global_mean_pool(x, batch)
            return self.output(x).squeeze(-1)

# Load Round 1 weights
paegat = PAEGATWithAttention().to(device)
state_dict = torch.load(f'{RESULTS_DIR}/paegat_v1_round1.pt', map_location=device)
paegat.load_state_dict(state_dict)
paegat.eval()
print("Loaded Round 1 weights")

# Attention analysis
attention_results = []

for g in boltz_graphs[:200]:
    try:
        heavy_len = g.heavy_len
        light_len = g.light_len
        antigen_len = g.antigen_len
        antigen_offset = heavy_len + light_len

        x = g.x.to(device)
        edge_index = g.edge_index.to(device)
        edge_attr = g.edge_attr.to(device)
        batch = torch.zeros(x.shape[0], dtype=torch.long).to(device)

        with torch.no_grad():
            pred, att_idx, att_w = paegat(x, edge_index, edge_attr, batch, return_attention=True)

        src = att_idx[0].cpu().numpy()
        dst = att_idx[1].cpu().numpy()
        weights = att_w.mean(dim=1).cpu().numpy()

        ha_mask = ((src < heavy_len) & (dst >= antigen_offset)) | \
                  ((dst < heavy_len) & (src >= antigen_offset))
        la_mask = ((src >= heavy_len) & (src < antigen_offset) & (dst >= antigen_offset)) | \
                  ((dst >= heavy_len) & (dst < antigen_offset) & (src >= antigen_offset))
        hl_mask = ((src < heavy_len) & (dst >= heavy_len) & (dst < antigen_offset)) | \
                  ((dst < heavy_len) & (src >= heavy_len) & (src < antigen_offset))

        attention_results.append({
            'pdb': g.pdb,
            'mean_att_ha': weights[ha_mask].mean() if ha_mask.sum() > 0 else np.nan,
            'mean_att_la': weights[la_mask].mean() if la_mask.sum() > 0 else np.nan,
            'mean_att_hl': weights[hl_mask].mean() if hl_mask.sum() > 0 else np.nan,
            'paegat_pred': pred.item(),
            'dockq_ha': g.y.item(),
            'iptm': g.iptm.item()
        })
    except Exception as e:
        pass

df_att = pd.DataFrame(attention_results)
df_att.to_csv(f'{RESULTS_DIR}/paegat_attention_analysis.tsv', sep='\t', index=False)
print(f"\nAttention analysis: {len(df_att)} complexes")
print(f"Mean attention H/Ag: {df_att['mean_att_ha'].mean():.4f}")
print(f"Mean attention L/Ag: {df_att['mean_att_la'].mean():.4f}")
print(f"Mean attention H/L:  {df_att['mean_att_hl'].mean():.4f}")

# Load V1 results for figures
df_r1 = pd.read_csv(f'{RESULTS_DIR}/paegat_v1_round1_results.tsv', sep='\t')
df_r2 = pd.read_csv(f'{RESULTS_DIR}/paegat_v1_round2_results.tsv', sep='\t')
df_r3 = pd.read_csv(f'{RESULTS_DIR}/paegat_v1_round3_results.tsv', sep='\t')

# Figure 1: Reliability curves
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
n_bins = 10
bins = np.linspace(0, 1, n_bins + 1)
bin_centers = (bins[:-1] + bins[1:]) / 2

for ax, df_r, title in zip(axes,
                            [df_r1, df_r2, df_r3],
                            ['Boltz-2 (Round 1)', 'OpenFold3 (Round 2)', 'Protenix (Round 3)']):
    labels = df_r['dockq_ha'].values
    preds = df_r['paegat_pred'].values
    iptms = df_r['iptm'].values
    acceptable = (labels >= 0.23).astype(float)

    paegat_acc, iptm_acc = [], []
    for i in range(n_bins):
        m_p = (preds >= bins[i]) & (preds < bins[i+1])
        m_i = (iptms >= bins[i]) & (iptms < bins[i+1])
        paegat_acc.append(acceptable[m_p].mean() if m_p.sum() > 0 else np.nan)
        iptm_acc.append(acceptable[m_i].mean() if m_i.sum() > 0 else np.nan)

    ax.plot([0,1],[0,1],'k--', label='Perfect', alpha=0.5)
    ax.plot(bin_centers, paegat_acc, 'b-o', label='PAE-GAT', linewidth=2, markersize=6)
    ax.plot(bin_centers, iptm_acc, 'r-s', label='ipTM', linewidth=2, markersize=6)
    ax.set_xlabel('Confidence', fontsize=12)
    ax.set_ylabel('Fraction Acceptable (DockQ≥0.23)', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=10)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_reliability_curves.pdf', dpi=300, bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/fig_reliability_curves.png', dpi=300, bbox_inches='tight')
print("Saved reliability curves")

# Figure 2: Ablation ladder
fig, ax = plt.subplots(figsize=(10, 6))
methods = ['ipTM\nbaseline', 'Random\nForest', 'GCN', 'GAT\n(no edge)', 'PAE-GAT\n(ours)']
ece_values = [0.3219, 0.2044, 0.1087, 0.1043, 0.0478]
colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']

bars = ax.bar(methods, ece_values, color=colors, alpha=0.8, edgecolor='black')
ax.set_ylabel('Expected Calibration Error (ECE)', fontsize=12)
ax.set_title('Ablation Study — PAE-GAT Components\n(Boltz-2 test set, trained on OF3+Protenix)', fontsize=12)
for bar, val in zip(bars, ece_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0, 0.40)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_ablation_ladder.pdf', dpi=300, bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/fig_ablation_ladder.png', dpi=300, bbox_inches='tight')
print("Saved ablation ladder")

# Figure 3: Scatter plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (conf, name, color) in zip(axes, [
    (df_r1['paegat_pred'].values, 'PAE-GAT', '#9467bd'),
    (df_r1['iptm'].values, 'ipTM', '#d62728')
]):
    labels = df_r1['dockq_ha'].values
    r, _ = stats.spearmanr(conf, labels)
    ax.scatter(conf, labels, alpha=0.3, s=10, color=color)
    ax.set_xlabel(f'{name} Score', fontsize=12)
    ax.set_ylabel('DockQ_HA', fontsize=12)
    ax.set_title(f'{name} vs DockQ_HA (Boltz-2)\nSpearman r = {r:.3f}', fontsize=12)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axhline(y=0.23, color='k', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_scatter_pred_vs_dockq.pdf', dpi=300, bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/fig_scatter_pred_vs_dockq.png', dpi=300, bbox_inches='tight')
print("Saved scatter plots")

# Figure 4: Attention by interface type
if len(df_att) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    interface_types = ['H/Ag\n(Heavy-Antigen)', 'L/Ag\n(Light-Antigen)', 'H/L\n(Heavy-Light)']
    att_means = [df_att['mean_att_ha'].mean(), df_att['mean_att_la'].mean(), df_att['mean_att_hl'].mean()]
    att_stds = [df_att['mean_att_ha'].std(), df_att['mean_att_la'].std(), df_att['mean_att_hl'].std()]
    colors_att = ['#d62728', '#1f77b4', '#2ca02c']
    ax.bar(interface_types, att_means, yerr=att_stds, color=colors_att, alpha=0.8,
           edgecolor='black', capsize=5)
    ax.set_ylabel('Mean Attention Weight', fontsize=12)
    ax.set_title('PAE-GAT Learned Attention by Interface Type\n(Round 1 model, Boltz-2 test set)', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/fig_attention_by_interface.pdf', dpi=300, bbox_inches='tight')
    plt.savefig(f'{FIG_DIR}/fig_attention_by_interface.png', dpi=300, bbox_inches='tight')
    print("Saved attention figure")

print("\nAll done. Runtime disconnecting...")
from google.colab import runtime
runtime.unassign()

Loading Boltz-2 test graphs...
Loaded 1109 Boltz-2 graphs
Loaded Round 1 weights

Attention analysis: 200 complexes
Mean attention H/Ag: 0.0049
Mean attention L/Ag: 0.0033
Mean attention H/L:  0.0015
Saved reliability curves
Saved ablation ladder
Saved scatter plots
Saved attention figure

All done. Runtime disconnecting...
